# 📝 그래프DB 개념 과제 LV2(응용): 음악 스트리밍 그래프

> LV1 에서 익힌 것들을 **조합**합니다. 인접 dict, 공통 이웃, 2홉 추천, 관계 집계, 트리플 변환, SPARQL 질의(패턴·UNION·NOT EXISTS·ASK·COUNT·속성 경로), 조인 vs 순회 vs 경로 비교(서술), 차수, 트리플을 LPG 로 옮기기, SPARQL 원문 읽기(서술), 그리고 그래프 시각화.

## 풀이 방법
1. 맨 위 **데이터 살펴보기** 셀을 먼저 실행하세요(`nodes`·`edges` 준비).
2. 각 문제의 **답안 셀**에 코드를 채우고 **자가채점 셀**로 확인하세요(✅ 통과!).
3. 10번은 정량·서술 결합, 13번은 서술 전용입니다. markdown 셀에 답을 적고 정답 노트북과 비교하세요.
4. 17번은 **자가채점이 없습니다**. 문제에 실린 완성 그래프와 같은 모양이 나오면 됩니다.

- 데이터: `data/music_graph.json`: 사용자-[들었다]->곡, 곡-[부른가수]->아티스트 의 방향 그래프입니다.

화이팅!

## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다. 채점에 쓰는 음악 그래프를 먼저 훑어봅니다.

In [ ]:
# [제공 코드] 음악 스트리밍 그래프를 불러옵니다
import json
from pathlib import Path
graph = json.loads(Path('data/music_graph.json').read_text(encoding='utf-8'))
nodes = graph['nodes']      # id -> {'name': 이름, 'type': 사용자/곡/아티스트}
edges = graph['edges']      # [{'relation': 들었다/부른가수, 'source': .., 'target': ..}]
print('노드', len(nodes), '개 · 관계', len(edges), '개')
for e in edges[:3]:
    print(e)

## 1. 인접 dict 만들기: 사용자가 들은 곡
**배경**: 순회를 빠르게 하려면 "각 사용자가 들은 곡"을 미리 모아 둡니다(인접 dict).

**요구사항**:
- `들었다` 관계로 인접 dict **`listened`** 를 만드세요. 키는 사용자 id, 값은 그 사용자가 들은 **곡 id 들의 집합(set)**.

**예시**: `listened['u1']` 은 민서가 들은 곡 id 집합입니다(곡 3개).

<details><summary>힌트</summary>

```text
접근방법:
- 들었다 엣지를 돌며 source 별로 target 을 집합에 모은다.

세부구현:
1. listened = {} 로 시작한다.
2. edges 중 relation 이 '들었다' 인 엣지마다, setdefault 로 source 키가 없으면 빈 집합을 만들고 그 집합에 target 을 add 한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert listened['u1'] == {'s1', 's2', 's3'}, \
    '민서가 들은 곡 3개를 집합으로 모았는지 확인하세요'
assert len(listened['u2']) == 3, '준우가 들은 곡도 3개입니다'
assert set(listened) == {'u1', 'u2', 'u3', 'u4'}, \
    '들었다 관계만 모았는지 확인하세요(곡 노드가 키로 들어오면 부른가수 관계까지 섞인 것입니다)'
print('✅ 통과!')

## 2. 공통 이웃: 두 사람이 함께 들은 곡
**배경**: 두 사용자의 곡 집합을 **교집합**하면 함께 들은 곡이 나옵니다(취향이 겹치는 정도).

**요구사항**:
- `민서(u1)` 와 `준우(u2)` 가 **둘 다 들은** 곡 id 집합을 **`common`** 에 담으세요(`listened` 활용).

**예시**: `common` 의 곡 이름은 `['Ditto', '좋은 날']` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- listened['u1'] 과 listened['u2'] 의 교집합(&)을 구한다.

세부구현:
1. listened['u1'] 과 listened['u2'] 의 교집합을 구해 common 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert common == {'s2', 's3'}, \
    '두 사람의 곡 집합을 교집합(&)했는지 확인하세요(합집합이면 결과가 커집니다)'
print('✅ 통과!')

## 3. 2홉 추천: 취향이 비슷한 사람이 들은 다른 곡
**배경**: "내가 들은 곡을 들은 다른 사람"이 들은 "내가 아직 안 들은 곡"을 추천합니다(2홉).

**요구사항**:
- `민서(u1)` 기준 2홉 추천 곡 id 집합 **`rec`** 를 만드세요.
  1. 민서가 들은 곡 집합 `my` 를 구한다.
  2. 민서가 아닌 사용자 중 `my` 와 겹치는 곡을 하나라도 들은 사람을 모은다.
  3. 그 사람들이 들은 곡을 모두 합치고, `my` 에 이미 있는 곡은 뺀다.

**예시**: `rec` 의 곡 이름은 `['Dynamite', 'OMG', '주저하는 연인들을 위해']` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 내 곡과 겹치는 사람을 찾고, 그들의 곡을 합친 뒤 내 곡을 뺀다.

세부구현:
1. my = listened['u1'] 로 둔다.
2. peers = u != 'u1' 이면서 listened[u] 와 my 의 교집합이 있는 사용자들.
3. peer 들이 들은 곡을 하나의 집합으로 합집합해 모은다.
4. 그 집합에서 내가 이미 들은 곡을 뺀다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert rec == {'s4', 's5', 's7'}, \
    '취향이 겹치는 사람들의 곡을 합친 뒤 내가 이미 들은 곡을 뺐는지 확인하세요'
print('✅ 통과!')

## 4. 관계 유형별 개수 집계
**배경**: 그래프에 어떤 관계가 얼마나 있는지 세면 구조가 한눈에 들어옵니다.

**요구사항**:
- `edges` 의 `relation` 별 개수를 사전 **`rel_count`** 에 담으세요(키=관계, 값=개수).

**예시**: `rel_count['들었다']` 는 **11**, `rel_count['부른가수']` 는 **7** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- collections.Counter 로 relation 값을 센다.

세부구현:
1. collections 에서 Counter 를 불러온다.
2. Counter 로 각 엣지의 relation 값을 세어 dict 로 만들어 rel_count 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert rel_count['들었다'] == 11, '엣지의 relation 값을 세었는지 확인하세요'
assert rel_count['부른가수'] == 7, '부른가수 관계도 빠짐없이 세었는지 확인하세요'
print('✅ 통과!')

## 5. 관계를 RDF 트리플로 변환
**배경**: 그래프 관계를 지식 그래프의 **트리플 `(주어, 술어, 목적어)`** 로 옮깁니다(id 대신 이름).

**요구사항**:
- `edges` 의 **모든** 관계를 `(출발 이름, 관계, 도착 이름)` 튜플로 바꿔 리스트 **`triples`** 에 담으세요. 이름은 `nodes[id]['name']` 으로 얻습니다.

**예시**: `triples[0]` 은 `('민서', '들었다', '밤편지')` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 각 엣지의 source·target id 를 이름으로 바꿔 (이름, 관계, 이름) 튜플로 만든다.

세부구현:
1. 리스트 컴프리헨션으로 각 엣지의 출발 이름·관계·도착 이름을 순서대로 담은 튜플을 만든다(이름은 nodes[id]['name']).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert triples[0] == ('민서', '들었다', '밤편지'), '첫 트리플부터 (출발 이름, 관계, 도착 이름) 순서로 담으세요'
assert len(triples) == 18, '엣지 18개를 모두 옮겼는지 확인하세요'
assert triples[-1] == ('주저하는 연인들을 위해', '부른가수', '잔나비'), \
    '마지막 엣지까지 같은 규칙으로 옮겼는지 확인하세요(같은 값을 반복해 채우면 안 됩니다)'
assert len({t[1] for t in triples}) == 2, '관계는 두 종류입니다. 한 종류만 담기지 않았는지 확인하세요'
print('✅ 통과!')

## 6. SPARQL 로 질의하기
**배경**: 교안에서 쓴 것과 같은 방식으로, 음악 그래프에 **SPARQL** 을 보내 "특정 곡을 들은 사람"을 찾습니다.

**요구사항**:
- 아래 제공된 **`music_graph`**(rdflib 그래프)와 **`QUERY_PREFIX`** 를 씁니다.
- `Ditto` 를 들은 사람들의 **이름**을 집합 **`ditto_listeners`** 에 담으세요.
- 패턴은 술어가 `ex:들었다`, 목적어가 `ex:Ditto` 이고 주어를 변수로 둡니다.
- 결과 값은 URI 이므로 **마지막 칸만 잘라 내고 밑줄을 공백으로 되돌려** 이름으로 만드세요.

**예시**: `Ditto` 를 들은 사람은 **2명**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 주어 자리를 변수로 둔 패턴 한 줄짜리 SELECT 질의를 만들어 실행하고, 결과의 주어만 모은다.

세부구현:
1. QUERY_PREFIX 뒤에 SELECT 와 WHERE 를 이어 붙여 질의문 문자열을 만든다.
2. WHERE 안에는 주어를 변수로, 술어와 목적어를 고정한 패턴 한 줄을 적고 마침표로 끝낸다.
3. music_graph 에 그 질의문을 실행하고, 각 행의 주어를 이름으로 되돌려 집합에 담는다.
```

</details>

In [ ]:
# [제공 코드] 음악 그래프를 rdflib 의 RDF 그래프로 옮긴다(실행만 하세요)
from rdflib import Graph, Namespace

EX = Namespace('http://example.org/music/')
music_graph = Graph()
music_graph.bind('ex', EX)
for e in edges:
    # URI 에는 공백을 쓸 수 없으므로 이름의 공백을 밑줄로 바꾼다
    subject = nodes[e['source']]['name'].replace(' ', '_')
    obj = nodes[e['target']]['name'].replace(' ', '_')
    music_graph.add((EX[subject], EX[e['relation']], EX[obj]))

# 질의문마다 앞에 붙일 PREFIX 는 한 번 만들어 두고 이어 붙여 쓴다
QUERY_PREFIX = 'PREFIX ex: <http://example.org/music/>\n'
print('그래프에 담긴 트리플 수:', len(music_graph))   # 출력: 18

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert ditto_listeners == {'민서', '준우'}, \
    '매치된 트리플의 주어(첫 칸)만 모았는지 확인하세요. 이름이지 id 가 아닙니다'
print('✅ 통과!')

## 7. 두 조건 중 하나로 고르기
**배경**: "밤편지 **또는** Ditto 를 들은 사람"은 패턴 한 줄로는 못 찾습니다. 교안에서 본 **`UNION`** 으로 두 패턴을 잇습니다.

**요구사항**:
- 앞 문제의 **`music_graph`** 와 **`QUERY_PREFIX`** 를 그대로 씁니다.
- `밤편지`(`ex:밤편지`) 또는 `Ditto`(`ex:Ditto`) 를 들은 사람 **이름**을 집합 **`union_listeners`** 에 담으세요.
- 두 패턴을 각각 중괄호로 감싸 `UNION` 으로 잇습니다.
- 결과 값은 URI 이므로 마지막 칸만 잘라 내고 밑줄을 공백으로 되돌려 이름으로 만드세요.

**예시**: 둘 중 하나라도 들은 사람은 **3명**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 곡마다 패턴을 하나씩 쓰고, 두 패턴을 UNION 으로 이어 둘 중 하나만 맞아도 남게 한다.

세부구현:
1. QUERY_PREFIX 뒤에 SELECT 와 WHERE 를 붙인다.
2. WHERE 안에 밤편지 패턴을 중괄호로 감싸고, UNION 을 적고, Ditto 패턴을 또 중괄호로 감싼다.
3. 실행 결과의 주어를 이름으로 되돌려 집합에 담는다(집합이라 중복은 저절로 사라진다).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert union_listeners == {'민서', '시우', '준우'}, \
    '두 패턴을 UNION 으로 이었는지 확인하세요. 한 곡만 조회하면 인원이 모자랍니다'
print('✅ 통과!')

## 8. 없는 것 찾기
**배경**: "Ditto 를 **듣지 않은** 사용자"는 패턴만으로는 적을 수 없습니다. 없다는 조건은 **`FILTER NOT EXISTS`** 로 씁니다.

**요구사항**:
- 곡을 하나라도 들은 사람 중, `Ditto` 를 들은 사실이 **없는** 사람 이름을 집합 **`no_ditto`** 에 담으세요.
- 먼저 `들었다` 관계로 후보를 만들고, 그 뒤에 `FILTER NOT EXISTS` 로 Ditto 를 들은 사람을 덜어 냅니다.
- 이어서 **`ASK`** 로 "하은(`ex:하은`)이 Ditto 를 들었는가"를 물어 참/거짓을 변수 **`haeun_heard_ditto`** 에 담으세요(`bool()` 로 감싸면 True/False 가 됩니다).

**예시**: Ditto 를 듣지 않은 사용자는 **2명**이고, 하은은 그중 하나입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 무언가를 들은 사람을 후보로 잡고, 그 후보 중 Ditto 를 들은 패턴이 없는 사람만 남긴다.

세부구현:
1. WHERE 안에 목적어를 변수로 둔 들었다 패턴 한 줄을 적어 후보를 만든다.
2. 그 아래에 FILTER NOT EXISTS 를 적고 중괄호 안에 Ditto 를 들은 패턴을 넣는다.
3. 실행 결과의 주어를 이름으로 되돌려 집합에 담는다.
4. ASK 질의문은 SELECT 없이 ASK 와 중괄호 패턴만 적고, 결과를 bool 로 바꿔 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert no_ditto == {'시우', '하은'}, \
    '후보를 먼저 만든 뒤 FILTER NOT EXISTS 로 덜어 냈는지 확인하세요'
assert haeun_heard_ditto is False, \
    'ASK 결과를 bool 로 바꿔 담았는지 확인하세요(하은은 Ditto 를 듣지 않았습니다)'
print('✅ 통과!')

## 9. 묶어서 세기: 가수별 재생 횟수
**배경**: "이 가수의 곡이 몇 번 재생됐나"를 구하려면 **사람 -> 곡 -> 가수**로 패턴 두 줄을 잇고, 가수마다 묶어 세야 합니다. `GROUP BY` 와 `COUNT` 가 그 일을 합니다.

**요구사항**:
- 결과를 사전 **`artist_plays`**(키 = 가수 이름, 값 = 재생 횟수 정수)에 담으세요.
- 개수는 `(COUNT(?변수) AS ?plays)` 꼴로 적고, `GROUP BY ?artist` 로 가수마다 묶습니다.
- `COUNT` 결과는 숫자 그대로 오지 않으므로 `int()` 로 바꿔 담으세요.

**예시**: 가수 **4명**이 나오고, 가장 많이 재생된 가수는 **4회**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 사람이 들은 곡을 찾고, 그 곡의 가수로 한 번 더 건너간 뒤 가수마다 묶어 센다.

세부구현:
1. WHERE 안에 들었다 패턴과 부른가수 패턴을 두 줄로 적고, 곡 자리에 같은 변수를 써서 잇는다.
2. SELECT 에 가수 변수와 개수를 적고, GROUP BY 로 가수마다 묶는다.
3. 결과를 돌며 가수 이름을 이름으로 되돌리고 개수는 정수로 바꿔 사전에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert artist_plays == {'뉴진스': 3, '방탄소년단': 3, '아이유': 4, '잔나비': 1}, \
    '두 패턴을 같은 곡 변수로 이었는지, 값을 int 로 바꿨는지 확인하세요'
print('✅ 통과!')

## 10. 조인 vs 순회 vs SPARQL 경로: 같은 답, 세 가지 길
**배경**: "민서가 들은 곡의 **아티스트**"를 구하는 세 가지 방법을 비교합니다. 표로 보면 두 표를 **조인(merge)** 해야 하고, 그래프로 보면 사용자→곡→아티스트로 **2홉 순회**하면 되며, SPARQL 로는 **속성 경로** 한 줄이면 됩니다.

**요구사항**:
- (제공된 `listen_df`·`sang_df` 사용) 두 DataFrame 을 `곡` 으로 **merge** 한 뒤 사용자가 `민서` 인 행의 `아티스트` 를 집합 **`artists_join`** 에 담으세요.
- (그래프 순회) 먼저 `부른가수` 관계로 사전 **`song_to_artist`**(키=**곡 id**, 값=**아티스트 id**)를 만드세요. `부른가수` 관계 **전부**가 들어가야 합니다(곡 하나에 아티스트 하나). 그런 다음 `listened['u1']` 의 각 곡을 이 사전으로 따라가 아티스트 **이름** 집합 **`artists_walk`** 를 만드세요.
- (SPARQL 경로) `music_graph` 에 **속성 경로**를 써서 `민서`(`ex:민서`)가 들은 곡의 아티스트 **이름** 집합 **`artists_path`** 를 만드세요. 2홉(`들었다` 다음 `부른가수`)을 슬래시로 이어 **패턴 한 줄**로 씁니다.
- 세 결과가 모두 같아야 합니다.

**예시**: 세 집합의 크기는 모두 **2** 이고 서로 같아야 합니다. `song_to_artist` 의 크기는 **7** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 조인: listen_df 와 sang_df 를 '곡' 으로 merge 하고 사용자=='민서' 필터.
- 순회: 곡 -> 아티스트 인접 dict 를 만들고, 민서의 곡마다 아티스트를 모은다.

세부구현:
1. listen_df 와 sang_df 를 '곡' 으로 merge 하고, 사용자가 '민서' 인 행의 '아티스트' 를 집합으로 모아 artists_join 에 담는다.
2. 부른가수 엣지로 song_to_artist(곡 id -> 아티스트 id) 를 만든다(곡마다 아티스트 하나).
3. 민서가 들은 곡마다 song_to_artist 로 아티스트 id 를 찾고, 그 id 를 이름으로 바꿔 집합으로 모아 artists_walk 에 담는다.
4. SPARQL 질의문의 WHERE 에 주어를 민서로 고정하고, 두 술어를 슬래시로 이어 한 줄로 적은 뒤 결과를 이름으로 되돌려 artists_path 에 담는다.
```

</details>

In [ ]:
# [제공 코드] 조인/순회 비교를 위해 두 관계를 각각 DataFrame 으로 준비(실행만 하세요)
import pandas as pd
listen_df = pd.DataFrame([
    {'사용자': nodes[e['source']]['name'], '곡': nodes[e['target']]['name']}
    for e in edges if e['relation'] == '들었다'])
sang_df = pd.DataFrame([
    {'곡': nodes[e['source']]['name'], '아티스트': nodes[e['target']]['name']}
    for e in edges if e['relation'] == '부른가수'])
display(listen_df.head())
display(sang_df.head())

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert song_to_artist['s5'] == 'a3' and len(song_to_artist) == 7, \
    'song_to_artist 는 부른가수 관계 7건 전부를 담아야 합니다(값은 아티스트 id)'
assert {nodes[song_to_artist[s]]['name'] for s in listened['u4']} == {'방탄소년단', '아이유', '잔나비'}, \
    '시우가 들은 곡의 아티스트가 맞는지 확인하세요(곡 id 를 아티스트 id 로 바꾼 뒤 이름을 꺼냅니다)'
assert artists_join == {'뉴진스', '아이유'}, \
    'merge 후 사용자가 민서인 행의 아티스트만 모았는지 확인하세요'
assert artists_walk == artists_join, \
    '두 방법의 결과가 다릅니다. song_to_artist 를 부른가수 관계 전부로 만들었는지 보세요'
assert artists_path == artists_join, \
    '속성 경로 결과가 다릅니다. 두 술어를 슬래시로 이어 한 줄로 적었는지 확인하세요'
print('✅ 통과!')

**서술**: 위 세 방법(조인·순회·경로)의 장단점을 3~4문장으로 적어 보세요. "연결을 더 깊이(3홉, 4홉) 따라가면 어느 쪽이 번거로워지는가?"를 꼭 언급하세요.

*(여기에 자신의 생각을 서술하세요)*

## 11. 곡별 청취 수 세기
**배경**: 노드로 **들어오는** 화살표 수가 그 노드의 인기입니다. 곡 노드로 들어오는 `들었다` 관계를 세면 곡마다 몇 명이 들었는지가 나옵니다.

**요구사항**:
- `들었다` 관계만 세어 사전 **`song_play_count`**(곡 id -> 그 곡을 들은 사용자 수)를 만드세요.
- 관계가 두 종류이니 `부른가수` 는 세지 않습니다(세면 아티스트 노드가 키로 섞입니다).

**예시**: 곡은 **7개**이고 값의 합은 `들었다` 관계 수와 같은 **11** 입니다. `song_play_count['s1']` 은 **2** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 들었다 엣지 하나가 도착(곡) 노드의 개수를 1 올린다.

세부구현:
1. 빈 dict 를 만든다.
2. edges 중 relation 이 들었다 인 것만 돌며 target 을 키로 개수를 1씩 더한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert set(song_play_count) == {'s1', 's2', 's3', 's4', 's5', 's6', 's7'}, \
    '들었다 관계만 세었는지 확인하세요(아티스트 id 가 키로 들어오면 부른가수까지 센 것입니다)'
assert sum(song_play_count.values()) == 11, \
    '값의 합은 들었다 관계 수(11)와 같아야 합니다'
assert song_play_count['s1'] == 2 and song_play_count['s4'] == 1, \
    '도착(target)을 키로 세었는지 확인하세요. source 로 세면 사용자별 청취 수가 됩니다'
print('✅ 통과!')

## 12. 트리플을 LPG 노드·관계로 옮기기
**배경**: 5번은 그래프를 트리플로 옮겼습니다(LPG -> RDF). 이번엔 **반대 방향**입니다. 트리플 하나를 받아 노드 두 개와 방향 관계 하나로 펼칩니다. 어떤 레이블을 붙일지는 **온톨로지 규칙**이 알려 줍니다.

**요구사항**:
- 사전 **`MUSIC_RULES`**(술어 -> (주어 타입, 목적어 타입))를 만드세요. `들었다` 는 (`'사용자'`, `'곡'`), `부른가수` 는 (`'곡'`, `'아티스트'`) 입니다.
- 함수 **`to_lpg(triple)`** 를 만드세요. 트리플 `(주어, 술어, 목적어)` 를 받아 **`(노드 A, 노드 B, 관계)`** 세 값을 돌려줍니다.
  - 노드는 `{'id': 이름, 'label': 타입}` 꼴의 dict 이고, 타입은 `MUSIC_RULES` 에서 꺼내 씁니다.
  - 관계는 `(주어, 술어, 목적어)` 튜플 그대로입니다.

**예시**: `to_lpg(('민서', '들었다', '밤편지'))` 는 `({'id': '민서', 'label': '사용자'}, {'id': '밤편지', 'label': '곡'}, ('민서', '들었다', '밤편지'))` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 술어로 규칙을 찾아 (주어 타입, 목적어 타입) 을 꺼낸 뒤 노드 두 개를 만든다.

세부구현:
1. MUSIC_RULES 에 술어 두 개와 각각의 타입 쌍을 담는다.
2. 함수 안에서 트리플을 세 값으로 풀고, 술어로 규칙을 찾아 두 타입을 꺼낸다.
3. id 와 label 을 담은 dict 두 개를 만들고, 트리플 자체를 관계로 함께 돌려준다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert MUSIC_RULES == {'들었다': ('사용자', '곡'), '부른가수': ('곡', '아티스트')}, \
    'MUSIC_RULES 는 술어 -> (주어 타입, 목적어 타입) 두 항목이어야 합니다'
assert to_lpg(('민서', '들었다', '밤편지')) == (
    {'id': '민서', 'label': '사용자'}, {'id': '밤편지', 'label': '곡'},
    ('민서', '들었다', '밤편지')), '노드 dict 의 키 이름(id·label)과 반환 순서를 지문과 대조하세요'
assert to_lpg(('Ditto', '부른가수', '뉴진스'))[1]['label'] == '아티스트', \
    '레이블을 고정해 두지 말고 술어에 맞는 규칙에서 꺼내 쓰세요'
print('✅ 통과!')

## 13. 진짜 SPARQL 질의 읽기 (서술)
**배경**: 6번에서 여러분이 쓴 SPARQL 은 공개 지식 그래프에도 그대로 통합니다. 아래는 실제 Wikidata 에 던지는 SPARQL 입니다.

```sparql
SELECT ?x ?xLabel WHERE {
  ?x wdt:P106 wd:Q639669 .    # P106 = 직업, Q639669 = 음악가
  ?x wdt:P27  wd:Q884 .       # P27 = 국적, Q884 = 대한민국
  SERVICE wikibase:label { bd:serviceParam wikibase:language "ko,en". }
}
```

**요구사항**: 아래 세 가지를 각각 한두 문장으로 markdown 셀에 적으세요.

1. 이 질의가 **무엇을 묻는지** 한 문장으로.
2. 두 줄에 **같은 `?x`** 를 쓴 것이 무슨 일을 하는지, 그리고 6번에서 여러분이 쓴 질의와 무엇이 다른지.
3. 이 질의가 이름 대신 `wd:Q884` 같은 **번호**를 쓰는 이유.

*(자가채점이 없습니다. 정답 노트북의 모범 서술과 견줘 보세요.)*

*(여기에 자신의 답을 서술하세요)*

## 14. 질의 결과로 새 그래프 만들기
**배경**: `SELECT` 은 표를 돌려줍니다. **`CONSTRUCT`** 는 찾은 것을 **트리플로 다시 적어** 새 그래프를 만듭니다. 2홉(사람→곡→가수)을 1홉(`좋아하는가수`)으로 접어 봅니다.

**요구사항**:
- `music_graph` 에 `CONSTRUCT` 질의를 보내, `?사람 ex:좋아하는가수 ?가수` 모양의 트리플을 만들고 새 `Graph` **`fav_graph`** 에 담으세요.
- `WHERE` 에는 속성 경로(`ex:들었다/ex:부른가수`)를 쓰면 한 줄로 끝납니다.
- 이어서 `민서`가 좋아하는 가수 **이름** 집합 **`minseo_fav`** 를 `fav_graph` 에서 뽑으세요.

**예시**: 새 그래프에는 트리플 **8개**가 담기고, 민서의 가수는 **2명**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- CONSTRUCT 는 결과가 행이 아니라 트리플이므로, 빈 그래프를 만들어 하나씩 add 한다.

세부구현:
1. CONSTRUCT 중괄호 안에 만들 트리플 모양을, WHERE 중괄호 안에 찾을 패턴을 적는다.
2. 빈 Graph 를 만들고 질의 결과를 돌며 add 한다.
3. fav_graph 에 SELECT 질의를 한 번 더 보내 민서의 가수만 모은다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(fav_graph) == 8, \
    'CONSTRUCT 결과를 빈 Graph 에 모두 add 했는지 확인하세요(그래프는 집합이라 중복은 접힙니다)'
assert minseo_fav == {'뉴진스', '아이유'}, \
    '새로 만든 fav_graph 에 질의했는지 확인하세요'
print('✅ 통과!')

## 15. 그래프를 고치는 질의
**배경**: SPARQL 은 읽기만 하는 언어가 아닙니다. `DELETE ... WHERE ...` 와 `INSERT ... WHERE ...` 로 **조건에 맞는 것을 한꺼번에** 고칠 수 있습니다. rdflib 에서는 `query` 가 아니라 **`update`** 로 보냅니다.

**요구사항**:
- 원본을 건드리지 않도록 `music_graph` 의 트리플을 새 `Graph` **`edit_graph`** 에 옮겨 담으세요.
- `DELETE ... WHERE ...` 로 **`부른가수` 관계를 전부** 지우고, 남은 트리플 수를 변수 **`after_delete`** 에 담으세요.
- 이어서 `INSERT ... WHERE ...` 로 **곡을 하나라도 들은 사람에게** `a ex:사용자` 표시를 붙이고, 그때의 트리플 수를 **`after_insert`** 에 담으세요.

**예시**: 지운 뒤에는 **11개**, 표시를 붙인 뒤에는 **15개**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 사본 그래프를 만든 뒤 update 를 두 번 보낸다.

세부구현:
1. 빈 Graph 를 만들고 music_graph 의 트리플을 하나씩 add 한다.
2. DELETE 중괄호와 WHERE 중괄호에 같은 패턴을 적어 update 로 보낸다.
3. INSERT 중괄호에는 붙일 트리플을, WHERE 중괄호에는 대상을 찾는 패턴을 적는다.
4. 각 단계 뒤에 len 으로 트리플 수를 센다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert after_delete == 11, \
    '부른가수 관계만 지웠는지, 사본에서 작업했는지 확인하세요'
assert after_insert == 15, \
    '곡을 들은 사람에게만 타입을 붙였는지 확인하세요(같은 트리플은 한 번만 들어갑니다)'
assert len(music_graph) == 18, '원본 music_graph 는 그대로여야 합니다(사본에서 고치세요)'
print('✅ 통과!')

## 16. 질의 안에서 값 만들기: BIND
**배경**: 지금까지는 URI 를 파이썬에서 잘라 이름으로 바꿨습니다. **`BIND`** 를 쓰면 그 일을 **질의 안에서** 할 수 있고, 만든 값에 바로 조건을 걸 수 있습니다.

**요구사항**:
- `music_graph` 의 모든 트리플에서 **목적어**를 이름으로 바꿔 `?name` 에 담으세요(`STRAFTER` 로 `music/` 뒤를 자르고, `REPLACE` 로 밑줄을 공백으로).
- 그중 이름에 **`나비`** 가 들어간 것만 남겨 집합 **`nabi_names`** 에 담으세요(`CONTAINS` 를 씁니다).
- 같은 방식으로 이름이 **`주`로 시작**하는 것만 남겨 집합 **`ju_names`** 에 담으세요(`REGEX` 와 `^` 를 씁니다).

**예시**: 두 집합 모두 **1개**씩입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 아무 트리플이나 잡는 패턴을 쓰고, BIND 로 목적어를 이름으로 바꾼 뒤 FILTER 를 건다.

세부구현:
1. WHERE 안에 주어·술어·목적어가 모두 변수인 패턴 한 줄을 적는다.
2. BIND 로 목적어를 문자열로 바꾸고 앞부분을 잘라 밑줄을 공백으로 바꾼 뒤 ?name 에 담는다.
3. FILTER 에 CONTAINS 를, 두 번째 질의에는 REGEX 를 쓴다.
4. 조건만 바꾼 질의를 하나 더 만들고, 결과의 name 값을 집합에 담는다(문자열이라 자를 필요가 없다).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert nabi_names == {'잔나비'}, \
    'BIND 로 만든 ?name 에 CONTAINS 를 걸었는지 확인하세요'
assert ju_names == {'주저하는 연인들을 위해'}, \
    'REGEX 의 ^ 는 문자열 시작을 뜻합니다'
print('✅ 통과!')

## 17. 음악 그래프를 그림으로 그리기
**배경**: 숫자로만 보던 그래프를 눈으로 보면 구조가 한눈에 들어옵니다. 교안에서 배운 `networkx` 로 이 음악 그래프를 그려 봅니다.

**요구사항**: 아래 **완성 그래프와 같은 모양**이 나오도록 그리세요(이 문제는 자가채점이 없습니다. 그림을 눈으로 견줘 보세요).

- `nodes`·`edges` 로 방향 그래프 **`G`** 를 만드세요(라벨은 `nodes[id]['name']`).
- 색은 노드의 `type` 에 따라: 사용자 `'#8ecae6'`(하늘색) · 곡 `'#ffb3c6'`(연분홍) · 아티스트 `'#ffe6a7'`(연노랑)
- `node_size=1200`, `arrows=True`, 제목은 `'음악 스트리밍 그래프'`
- 긴 라벨이 그림 밖으로 잘리지 않도록 마지막에 여백을 `plt.margins(0.12)` 로 넓히세요.
- 배치는 `spring_layout` 에 **`seed=19`** 를 주어 고정하세요(같은 그림이 나오도록). 그림 크기는 `figsize=(10, 7.5)` 로 두면 아래와 가장 비슷합니다.
- 한글 라벨이 깨지지 않게, 아래 **제공 폰트 셀을 먼저 실행**하고 `nx.draw` 에 `font_family=KOREAN_FONT` 를 넘기세요.

**완성 그래프**

<img src="images/과제/lv2_music_graph.png" width="820">

<details><summary>힌트</summary>

```text
접근방법:
- 노드를 넣을 때 type 도 함께 저장해 두면 색 리스트를 만들기 쉽다.

세부구현:
1. nx.DiGraph 를 만들고 nodes 를 돌며 라벨과 종류를 속성으로 넣어 노드를 추가한다.
2. edges 를 돌며 source 에서 target 으로 관계를 추가한다.
3. 종류 -> 색 dict 를 만들고, 그래프의 노드 순서대로 색 리스트를 만든다.
4. 새 도화지를 열고 spring_layout 배치로 그린 뒤 제목을 붙이고 보여 준다.
```

</details>

In [ ]:
# [제공 코드] 한글 폰트 설정(실행만 하세요)
import platform
import matplotlib.pyplot as plt

# 그림에 한글 라벨이 들어가므로 폰트를 먼저 잡는다. 안 잡으면 글자가 네모로 나온다
# 한글 폰트: 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT    # 이후 모든 그림에 이 폰트가 적용된다
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

In [ ]:
# 여기에 코드를 작성하세요

## 18. 원본 지식 그래프에 직접 물어보기
**배경**: 지금까지는 우리가 만든 작은 그래프에 물었습니다. **같은 SPARQL 이 공개 지식 그래프에도 그대로 통합니다.** 교안 5절에서 본 Wikidata 에 직접 질의해 봅니다.

**요구사항**:

1. `"아이유"` 라는 **한국어 이름표**를 가진 항목을 찾으세요. 그런데 이름으로만 찾으면 가수가 아닌 것도 딸려 옵니다. **유형이 사람(`wd:Q5`)** 인 것만 남겨 그 항목 번호를 **`iu_qid`** 에 담으세요(`'Q20145'` 같은 문자열).
2. `"방탄소년단"` 의 **원산지(`wdt:P495`)** 를 한국어 이름으로 받아 **`bts_origin`** 에 담으세요. 사람이 아니라 **단체**라 국적(`P27`)이 아니라 원산지를 씁니다.

**예시**: 이름표로만 찾으면 후보가 **2개**인데, 사람으로 좁히면 **1개**만 남습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 라벨은 언어 태그를 붙여 그대로 비교할 수 있다. 유형은 P31 로 좁힌다.

세부구현:
1. WHERE 안에 라벨 패턴을 적되 값 뒤에 언어 태그를 붙인다.
2. 그 아래에 유형 패턴을 한 줄 더 적어 사람만 남긴다.
3. 결과의 항목 주소에서 마지막 칸만 잘라 번호만 담는다.
4. 두 번째 질의는 라벨로 단체를 찾고, 원산지로 한 걸음 간 뒤 그 이름을 받는다.
```

</details>

In [ ]:
# [제공 코드] 공개 지식 그래프에 질의를 보낼 준비(실행만 하세요)
from SPARQLWrapper import SPARQLWrapper, JSON

QLEVER = 'https://qlever.cs.uni-freiburg.de/api/wikidata'
# 공개 API 에는 누가 보내는지 밝힌다. 헤더는 한글을 못 담으니 영문으로 적는다
USER_AGENT = 'EncoreAICampus-day27/1.0 (classroom practice)'
WD_PREFIX = ('PREFIX wd: <http://www.wikidata.org/entity/>\n'
             'PREFIX wdt: <http://www.wikidata.org/prop/direct/>\n'
             'PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n')

def run_sparql(query, endpoint=QLEVER):
    """SPARQL 질의문을 공개 엔드포인트에 보내고 결과 행 리스트를 돌려준다.

    query    : 보낼 SPARQL 질의문
    endpoint : 질의를 받아 줄 주소(생략하면 QLever)
    """
    client = SPARQLWrapper(endpoint, agent=USER_AGENT)
    client.setQuery(query)
    client.setReturnFormat(JSON)
    client.setTimeout(60)
    return client.query().convert()['results']['bindings']

# 인터넷이 막힌 환경에서도 나머지 문제를 풀 수 있게, 연결되는지 먼저 확인해 둔다
try:
    run_sparql(WD_PREFIX + 'SELECT ?n WHERE { wd:Q20145 rdfs:label ?n . } LIMIT 1')   # Q20145 = 아이유
    ONLINE = True
except Exception as error:
    ONLINE = False
    print('연결 실패:', error)
print('인터넷 연결:', ONLINE)

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
if not ONLINE:
    print('⏭️ 인터넷에 연결할 수 없어 이 문제는 채점을 건너뜁니다')
else:
    assert iu_qid == 'Q20145', \
        '유형이 사람(wd:Q5)인 것만 남겼는지 확인하세요. 이름표만으로 찾으면 다른 항목이 섞입니다'
    assert bts_origin == '대한민국', \
        '단체는 국적(P27)이 아니라 원산지(P495)를 씁니다'
    print('✅ 통과!')

---
수고했어요! LV2 에서 인접 dict·공통 이웃·2홉 추천·집계·트리플을 **조합**하고, SPARQL 로 `UNION`·`FILTER NOT EXISTS`·`ASK`·`COUNT`·속성 경로까지 질의해 봤습니다. 조인과 순회와 경로를 나란히 비교했고, 트리플을 LPG 로 옮기고 진짜 SPARQL 원문도 읽었습니다. 마지막으로 그래프를 **그림**으로도 그렸고, 마지막에는 **공개 지식 그래프 원본**에 직접 질의해 봤습니다.